## 00. Quick Start


In [ ]:
print('Concept Portfolio V2 Lab — 기본 MODE는 MOCK입니다.')
print('LIVE는 06→28 순서의 staged 검증 후 46의 가드를 직접 켜야 합니다.')

## 01. Environment


In [ ]:
import os, sys, json
from pathlib import Path
from IPython.display import display
SEARCH_ROOTS = [Path.cwd(), *Path.cwd().parents]
AI_ROOT = next((p for p in SEARCH_ROOTS if (p / 'app').is_dir()), None)
if AI_ROOT is None: AI_ROOT = next((p / 'ai' for p in SEARCH_ROOTS if (p / 'ai' / 'app').is_dir()))
if str(AI_ROOT) not in sys.path: sys.path.insert(0, str(AI_ROOT))
print({'python': sys.version.split()[0], 'aiRoot': str(AI_ROOT)})

## 02. MODE


In [ ]:
MODE = 'MOCK'  # MOCK | REPLAY | LIVE
RECORDINGS_DIR = AI_ROOT / 'recordings' / 'concept_portfolio_v2'
print({'mode': MODE, 'liveExternalOperationsEnabled': MODE == 'LIVE'})

## 03. Environment Check


In [ ]:
LIVE_ENV_KEYS = ['AI_PROVIDER', 'AI_API_KEY', 'AI_MODEL', 'MOLEG_API_KEY', 'LEGAL_REGISTRY_VERSION']
env_status = {key: bool(os.getenv(key)) for key in LIVE_ENV_KEYS}
print(env_status if MODE == 'LIVE' else {'mode': MODE, 'message': '외부 환경변수 불필요'})

## 04. Schema Preflight


In [ ]:
from app.concept_portfolio_v2 import ConceptPortfolioEngine, ProviderGateway, ProviderMode
from app.concept_portfolio_v2.adapters import CurrentLegalAdapter
from app.concept_portfolio_v2.diagnostics.notebook_view import *
gateway = ProviderGateway(MODE, recordings_dir=RECORDINGS_DIR)
engine = ConceptPortfolioEngine(MODE, gateway=gateway)
schema_preflight = engine.schema_preflight_report()
display(show_schema_preflight(schema_preflight))
assert schema_preflight.status == 'PASS' and schema_preflight.providerCalls == 0

## 05. Input


In [ ]:
TEST_INPUT = {
    'ideaOverview': '개인 맞춤형 소량 식재료를 제공하고 남은 재료 활용 레시피를 안내하는 서비스',
    'problem': '1~2인 가구가 큰 포장 단위 때문에 식재료를 남기고 음식물 쓰레기가 발생한다',
    'targetUsers': '요리할 시간이 적고 낭비를 줄이고 싶은 1~2인 가구',
}
MAX_CONCEPTS = 5

## 06. Idea Brief Derivation


In [ ]:
engine._reset()
seed = engine.seed_adapter.adapt(TEST_INPUT)
idea_context = await engine.derive_idea_brief(seed)
print({'ideaCallComplete': True, 'interpretationPresent': bool(seed.interpretation)})

## 07. Safety


In [ ]:
display(idea_context.safetyReview.model_dump(mode='json'))
assert idea_context.safetyReview.passed

## 08. AI가 이해한 아이디어


In [ ]:
display(show_idea_interpretation(idea_context))

## 09. Readiness / Summary / commitments


In [ ]:
display(show_idea_readiness(idea_context))

## 10. Seed Analysis


In [ ]:
analysis = await engine.analyze_seed(seed)
display(show_seed_analysis(analysis))

## 11. Generic Opportunity Kernel


In [ ]:
display(analysis.opportunityKernel.model_dump(mode='json'))

## 12. Design Space


In [ ]:
display(show_design_space(analysis))

## 13. Generate and Adaptively Replenish Plan Pool


In [ ]:
plan_validation = await engine.prepare_portfolio_plans(seed, analysis, max_concepts=MAX_CONCEPTS)
plans = engine._last_plan_pool
print({'totalPlans': len(plans), 'planningRounds': plan_validation.planningRounds,
       'replenishmentRequested': plan_validation.replenishmentRequested})

## 14. Plan Count / Adaptive Replenishment Check


In [ ]:
display(show_plan_pool_status(engine._last_plan_pool_status))
display({'planningRounds': plan_validation.planningRounds,
         'replenishmentRequested': plan_validation.replenishmentRequested,
         'adaptiveReplenishmentUsed': plan_validation.planningRounds > 1})

## 15. Korean Plan Display


In [ ]:
display(show_portfolio_plans(plan_validation.acceptedPlans + plan_validation.reservePlans))

## 16. Plan Lock/Intent Validation


In [ ]:
display({'accepted': [p.planId for p in plan_validation.acceptedPlans],
         'rejected': [p.model_dump(mode='json') for p in plan_validation.rejectedPlans]})

## 17. Portfolio Family / Variant / Distinct


In [ ]:
display(show_plan_diversity(plan_validation.diversity))

## 18. Selected + Reserve Plans


In [ ]:
selected_plans = plan_validation.acceptedPlans
reserve_plans = plan_validation.reservePlans
display(show_portfolio_plans(selected_plans + reserve_plans))
display({'selected': [p.planId for p in selected_plans], 'reserve': [p.planId for p in reserve_plans]})

## 19. Candidate 1


In [ ]:
candidate_one = await engine.expand_plan(seed, selected_plans[0], 1) if selected_plans else None
display(show_candidates([candidate_one]) if candidate_one else [])

## 20. Candidate 1 Korean/Governance


In [ ]:
candidate_one_reports = []
display({'candidateId': candidate_one.candidateId if candidate_one else None,
         'status': 'PENDING_FULL_CANDIDATE_RECOVERY'})

## 21. Candidate 1 Actual Generic Descriptor


In [ ]:
display(show_concept_descriptors([candidate_one]) if candidate_one else [])

## 22. Candidate 1 Fidelity


In [ ]:
display({'candidateId': candidate_one.candidateId if candidate_one else None,
         'fidelity': '전체 Candidate Recovery 단계에서 semantic fallback 포함 검증'})

## 23. Remaining Candidates


In [ ]:
remaining_candidates = []
for i, plan in enumerate(selected_plans[1:], 2):
    remaining_candidates.append(await engine.expand_plan(seed, plan, i))
candidate_drafts = ([candidate_one] if candidate_one else []) + remaining_candidates
display(show_candidates(candidate_drafts))

## 24. Candidate Actual Generic Descriptors


In [ ]:
display(show_concept_descriptors(candidate_drafts))

## 25. Candidate Recovery / Portfolio Relations


In [ ]:
candidate_preparation = await engine.prepare_candidate_portfolio(
    seed, plan_validation, max_concepts=MAX_CONCEPTS, initial_candidates=candidate_drafts)
candidates = candidate_preparation.candidates
candidate_reports = candidate_preparation.reports
display(show_candidate_recovery(candidate_preparation))
candidate_pairwise = [engine.compare_candidates(candidates[i], candidates[j])
                      for i in range(len(candidates)) for j in range(i + 1, len(candidates))]
display(show_plan_diversity(candidate_pairwise))

## 26. Structural Legal Precheck + C1 Legal Fact Pattern


In [ ]:
prechecks = [engine.legal_precheck(item) for item in candidates]
display(show_legal_precheck(prechecks))
display(show_legal_fact_pattern(candidates[0].candidate, seed) if candidates else [])

## 27. Legal C1 Evidence Summary


In [ ]:
legal_adapter = CurrentLegalAdapter()
legal_c1_input = legal_adapter.task_input(candidates[0].candidate, seed) if candidates else None
display({'candidateId': candidates[0].candidateId if candidates else None,
         'externalFacts': legal_c1_input['externalFactContext']['facts'] if legal_c1_input else [],
         'note': '공식 근거 수와 allowed index는 Full Legal 응답/실패 diagnostics에서 확인'})

## 28. Full Legal C1


In [ ]:
RUN_FULL_LEGAL_C1 = False  # staged LIVE 확인 후 True
legal_one = None
if RUN_FULL_LEGAL_C1 and candidates:
    try:
        legal_one = await engine.review_legal_candidate(seed, candidates[0])
        display(show_legal_result([legal_one]))
    except Exception:
        display(show_legal_failure(candidates[0].candidateId, engine.gateway))
else:
    print('SKIPPED — RUN_FULL_LEGAL_C1=True로 명시해야 실행됩니다.')

## 29. Remaining Legal


In [ ]:
RUN_REMAINING_LEGAL = False
legal_remaining = []
if RUN_REMAINING_LEGAL and legal_one and len(candidates) > 1:
    legal_remaining = await engine.review_legal(seed, candidates[1:])
display(show_legal_result(legal_remaining) if legal_remaining else {'status': 'SKIPPED'})
display({'Plan Selected': len(selected_plans),
         'Candidate Generated': candidate_preparation.candidateGenerated,
         'Candidate Accepted': len(candidates),
         'Legal Reviewed': len(([legal_one] if legal_one else []) + legal_remaining)})

## 30. Redesign


In [ ]:
legal_initial = ([legal_one] if legal_one else []) + legal_remaining
portfolio, legal_all, required_inputs, redesigned_count, replanned_count = ([], legal_initial, [], 0, 0)
if len(legal_initial) == len(candidates) and candidates:
    portfolio, legal_all, required_inputs, redesigned_count, replanned_count = await engine.resolve_legal(
        seed, candidate_preparation.usedPlans, candidates, legal_initial)
print({'Plan Selected': len(selected_plans),
       'Candidate Generated': candidate_preparation.candidateGenerated,
       'Candidate Accepted': len(candidates),
       'Legal Accepted': sum(item.route.value == 'ACCEPT' for item in legal_all),
       'Legal Redesigned': redesigned_count, 'Legal Replanned': replanned_count,
       'Final Portfolio': len(portfolio)})

## 31. Replan


In [ ]:
display(show_replan(type('PortfolioView', (), {'concepts': portfolio})()))
print({'replanned': replanned_count, 'reserveAvailable': len(reserve_plans)})

## 32. Final Portfolio


In [ ]:
display(show_final_portfolio(type('PortfolioView', (), {'concepts': portfolio})()) if portfolio else {'status': 'LEGAL_PENDING'})

## 33. Unresolved Candidate Summary


In [ ]:
display(required_inputs if required_inputs else {'unresolved': []})

## 34. Manual Concept Selection


In [ ]:
SELECTED_CANDIDATE_ID = portfolio[0].candidateId if portfolio else None  # 사용자가 수정
selected_concept = next((item for item in portfolio if item.candidateId == SELECTED_CANDIDATE_ID), None)
print({'selectedCandidateId': SELECTED_CANDIDATE_ID})

## 35. 7 Hypotheses


In [ ]:
hypotheses = engine.build_or_load_current_hypothesis_contract(selected_concept) if selected_concept else []
display(show_hypotheses(hypotheses))

## 36. Confirm / Edit


In [ ]:
CONFIRM_ALL_PROPOSED = False
HYPOTHESIS_EDITS = {
    # 'PRICE': '월 17,900원',
}
confirmed_hypotheses = engine.confirm_hypotheses(
    hypotheses, HYPOTHESIS_EDITS, confirm_all_proposed=CONFIRM_ALL_PROPOSED) if hypotheses else []
display(show_hypotheses(confirmed_hypotheses))

## 37. Actual Delta Legal


In [ ]:
RUN_DELTA_LEGAL = False
delta_legal_result = None
if RUN_DELTA_LEGAL and selected_concept and any(h.deltaLegalRequired for h in confirmed_hypotheses):
    delta_legal_result = await engine.review_delta_legal(seed, selected_concept, confirmed_hypotheses)
    confirmed_hypotheses = engine.mark_delta_legal_reviewed(confirmed_hypotheses, delta_legal_result)
display(delta_legal_result.model_dump(mode='json') if delta_legal_result else {'status': 'NOT_REQUIRED_OR_SKIPPED'})

## 38. Market Seed


In [ ]:
handoff = None
if selected_concept and legal_all:
    handoff = engine.build_downstream_handoff(seed, selected_concept, confirmed_hypotheses, legal_all)
display(handoff.marketAnalysisSeedSnapshot if handoff else {'status': 'NOT_READY'})

## 39. Marketing Source


In [ ]:
display(handoff.marketingSourceSnapshot if handoff else {'status': 'NOT_READY'})

## 40. Contract Compatibility


In [ ]:
display(show_downstream_handoff(handoff) if handoff else {'contract': 'NOT_READY'})

## 41. Trace


In [ ]:
display(show_trace(engine.trace))

## 42. Provider/Legal Usage


In [ ]:
display(show_provider_usage(engine.gateway.usage))
print('상위 외부 작업 수는 내부 AI/MOLEG 네트워크 호출 수와 동일하다고 주장하지 않습니다.')

## 43. Replay Manifest


In [ ]:
display(show_replay_manifest(engine.gateway))

## 44. One-click MOCK


In [ ]:
mock_result = await ConceptPortfolioEngine('MOCK').run_full(
    TEST_INPUT, max_concepts=MAX_CONCEPTS, auto_confirm_hypotheses=True)
display(show_run_summary(mock_result))
assert mock_result.handoff and mock_result.handoff.contractStatus == 'CONTRACT_PASS'

## 45. One-click REPLAY


In [ ]:
RUN_ONE_CLICK_REPLAY = False
replay_result = None
if RUN_ONE_CLICK_REPLAY:
    replay_gateway = ProviderGateway('REPLAY', recordings_dir=RECORDINGS_DIR)
    replay_result = await ConceptPortfolioEngine('REPLAY', gateway=replay_gateway).run_full(
        TEST_INPUT, max_concepts=MAX_CONCEPTS, auto_confirm_hypotheses=False)
display(show_run_summary(replay_result) if replay_result else {'status': 'SKIPPED'})

## 46. One-click LIVE


In [ ]:
RUN_ONE_CLICK_LIVE = False
live_result = None
if RUN_ONE_CLICK_LIVE:
    assert MODE == 'LIVE', 'MODE=LIVE를 먼저 명시하세요.'
    assert RUN_FULL_LEGAL_C1 and RUN_REMAINING_LEGAL, 'staged LIVE Legal 확인이 먼저입니다.'
    live_result = await engine.run_full(TEST_INPUT, max_concepts=MAX_CONCEPTS,
                                        auto_confirm_hypotheses=False)
display(show_run_summary(live_result) if live_result else {'status': 'SKIPPED'})